# Crypto Crash Analysis & Recovery Metrics (Feb 2026 Case Study)


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Crypto Crash Forensic Analysis: Feb 2026 vs Historical Events

This notebook analyzes the February 2026 crypto crash (Bitcoin -21.7%, Ethereum -28.5%) and compares recovery patterns to 2022 and 2018 crashes. Built for traders studying market resilience and recovery trajectories.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Historical crash event definitions (price drop % and approximate dates)
crash_events = {
    'Feb 2026': {'btc_drop': -21.7, 'eth_drop': -28.5, 'peak_date': '2026-02-01', 'trough_date': '2026-02-15'},
    'May 2022': {'btc_drop': -14.3, 'eth_drop': -19.2, 'peak_date': '2022-05-01', 'trough_date': '2022-05-12'},
    'Jan 2018': {'btc_drop': -32.8, 'eth_drop': -38.5, 'peak_date': '2018-01-07', 'trough_date': '2018-02-06'}
}

print('Crash Events Summary:')
for event, metrics in crash_events.items():
    print(f"{event}: BTC {metrics['btc_drop']}%, ETH {metrics['eth_drop']}%")

## Generate Synthetic Historical Data

For demonstration, we'll create realistic price series centered on each crash event. In production, replace with live data from CoinGecko, Binance, or similar API.

In [ ]:
def generate_crash_scenario(peak_price, drop_pct, peak_date_str, trough_date_str, days_recovery=120):
    """
    Generate synthetic price data around a crash event.
    peak_price: starting price at peak
    drop_pct: negative % drop to trough
    peak_date_str: date of peak (YYYY-MM-DD)
    trough_date_str: date of trough (YYYY-MM-DD)
    days_recovery: days of recovery data post-trough
    """
    peak_date = datetime.strptime(peak_date_str, '%Y-%m-%d')
    trough_date = datetime.strptime(trough_date_str, '%Y-%m-%d')
    
    # Pre-crash data (30 days before peak)
    pre_dates = [peak_date - timedelta(days=30-i) for i in range(30)]
    pre_prices = np.linspace(peak_price * 0.98, peak_price, 30) + np.random.normal(0, peak_price*0.01, 30)
    
    # Crash phase
    crash_days = (trough_date - peak_date).days
    crash_dates = [peak_date + timedelta(days=i) for i in range(crash_days)]
    trough_price = peak_price * (1 + drop_pct / 100)
    crash_prices = np.linspace(peak_price, trough_price, crash_days) + np.random.normal(0, peak_price*0.005, crash_days)
    
    # Recovery phase
    recovery_dates = [trough_date + timedelta(days=i) for i in range(days_recovery)]
    recovery_prices = trough_price + (peak_price - trough_price) * (1 - np.exp(-np.arange(days_recovery)/60)) + np.random.normal(0, peak_price*0.01, days_recovery)
    
    dates = pre_dates + crash_dates + recovery_dates
    prices = list(pre_prices) + list(crash_prices) + list(recovery_prices)
    
    return pd.DataFrame({'date': dates, 'price': prices})

# Generate data for all events
btc_scenarios = {}
eth_scenarios = {}

for event, metrics in crash_events.items():
    # BTC starting at $40k, $45k, $60k depending on era
    btc_peak = 40000 if '2018' in event else (45000 if '2022' in event else 65000)
    eth_peak = 800 if '2018' in event else (3000 if '2022' in event else 3500)
    
    btc_scenarios[event] = generate_crash_scenario(
        btc_peak, metrics['btc_drop'], metrics['peak_date'], metrics['trough_date']
    )
    eth_scenarios[event] = generate_crash_scenario(
        eth_peak, metrics['eth_drop'], metrics['peak_date'], metrics['trough_date']
    )

print(f"Generated BTC data for {len(btc_scenarios)} events")
print(f"Generated ETH data for {len(eth_scenarios)} events")

## Calculate Key Metrics: Drawdown & Recovery Timeline

In [ ]:
def compute_drawdown_metrics(prices):
    """
    Compute cumulative maximum, drawdown %, and recovery time.
    """
    cummax = np.maximum.accumulate(prices)
    drawdown = (prices - cummax) / cummax * 100
    max_drawdown = drawdown.min()
    
    # Time to recovery (first date when price >= previous peak)
    recovery_idx = np.where(prices >= cummax)[0]
    days_to_recovery = recovery_idx[-1] if len(recovery_idx) > 0 else len(prices)
    
    return {
        'max_drawdown': max_drawdown,
        'days_to_recovery': days_to_recovery,
        'drawdown_series': drawdown
    }

metrics_summary = []

for event in crash_events.keys():
    btc_metrics = compute_drawdown_metrics(btc_scenarios[event]['price'].values)
    eth_metrics = compute_drawdown_metrics(eth_scenarios[event]['price'].values)
    
    metrics_summary.append({
        'Event': event,
        'BTC Max Drawdown (%)': round(btc_metrics['max_drawdown'], 2),
        'BTC Days to Recovery': btc_metrics['days_to_recovery'],
        'ETH Max Drawdown (%)': round(eth_metrics['max_drawdown'], 2),
        'ETH Days to Recovery': eth_metrics['days_to_recovery']
    })

metrics_df = pd.DataFrame(metrics_summary)
print('\n=== Crash Recovery Metrics ===\n')
print(metrics_df.to_string(index=False))

## Visualization: Crash & Recovery Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: BTC Price Series (All Events)
ax = axes[0, 0]
for event in crash_events.keys():
    df = btc_scenarios[event]
    ax.plot(df['date'], df['price'], label=event, linewidth=2, alpha=0.8)
ax.set_title('Bitcoin Price: Crash Events Comparison', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: ETH Price Series (All Events)
ax = axes[0, 1]
for event in crash_events.keys():
    df = eth_scenarios[event]
    ax.plot(df['date'], df['price'], label=event, linewidth=2, alpha=0.8)
ax.set_title('Ethereum Price: Crash Events Comparison', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Drawdown % (BTC)
ax = axes[1, 0]
for event in crash_events.keys():
    prices = btc_scenarios[event]['price'].values
    metrics = compute_drawdown_metrics(prices)
    ax.plot(metrics['drawdown_series'], label=event, linewidth=2, alpha=0.8)
ax.set_title('Bitcoin Drawdown % Over Time', fontsize=12, fontweight='bold')
ax.set_xlabel('Days from Peak')
ax.set_ylabel('Drawdown (%)')
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Drawdown % (ETH)
ax = axes[1, 1]
for event in crash_events.keys():
    prices = eth_scenarios[event]['price'].values
    metrics = compute_drawdown_metrics(prices)
    ax.plot(metrics['drawdown_series'], label=event, linewidth=2, alpha=0.8)
ax.set_title('Ethereum Drawdown % Over Time', fontsize=12, fontweight='bold')
ax.set_xlabel('Days from Peak')
ax.set_ylabel('Drawdown (%)')
ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nVisualization complete.')

## Volatility Analysis During Crashes

Compute rolling volatility to understand price stability before, during, and after each crash.

In [ ]:
def compute_rolling_volatility(prices, window=14):
    """
    Compute rolling standard deviation as volatility proxy.
    """
    return pd.Series(prices).rolling(window=window).std().values

volatility_data = []

for event in crash_events.keys():
    btc_prices = btc_scenarios[event]['price'].values
    eth_prices = eth_scenarios[event]['price'].values
    
    btc_vol = compute_rolling_volatility(btc_prices, window=7)
    eth_vol = compute_rolling_volatility(eth_prices, window=7)
    
    # Segment: pre-crash (first 30 days), crash (next ~15 days), recovery (remaining)
    pre_crash_vol_btc = np.nanmean(btc_vol[:30])
    crash_vol_btc = np.nanmean(btc_vol[30:45])
    recovery_vol_btc = np.nanmean(btc_vol[45:])
    
    pre_crash_vol_eth = np.nanmean(eth_vol[:30])
    crash_vol_eth = np.nanmean(eth_vol[30:45])
    recovery_vol_eth = np.nanmean(eth_vol[45:])
    
    volatility_data.append({
        'Event': event,
        'BTC Pre-Crash Vol': round(pre_crash_vol_btc, 4),
        'BTC Crash Vol': round(crash_vol_btc, 4),
        'BTC Recovery Vol': round(recovery_vol_btc, 4),
        'ETH Pre-Crash Vol': round(pre_crash_vol_eth, 4),
        'ETH Crash Vol': round(crash_vol_eth, 4),
        'ETH Recovery Vol': round(recovery_vol_eth, 4)
    })

vol_df = pd.DataFrame(volatility_data)
print('\n=== Volatility Analysis (7-day rolling std) ===\n')
print(vol_df.to_string(index=False))
print('\nNote: Higher volatility during crash phase indicates increased price instability.')

---

## Reference

[read the guide](https://protraderdaily.com/crypto/crypto-market-analysis-february-2026)
